## Задача 1

Допустим, у нас существует таблица с возвратами товаров от клиента.

Таблица returns (возвраты):
| return_id | order_item_id | return_date | reason              |
|--------------|--------------------|------------------|------------------- |
| 7001         | 5001                 | 2023-04-20  | Не тот размер|
| 7002         | 5003                 | 2023-05-15  | Брак                  |
| 7003         | 12345               | 2024-01-05  | бракованный |
| …

Здесь в поле reason клиент в произвольной форме указывает причину возврата. С помощью языка запросов sql вычислите оценку снизу и сверху доли возвратов с причиной «брак».


Для работы с SQL в Colab потребуется SQLite, он тут единственны работает

Но также я буду приводить примеры query-запросов для psql

Подгрузим необходимые библиотеки для работы с SQLite

In [1]:
!pip install -q ipython-sql
!pip install -q prettytable==3.9.0

Загрузим расширение sql для колаба, которое нам даст возможность выполнять query-запросы в ячейках

In [2]:
%load_ext sql

Определяем нашу СУБД

In [3]:
%sql sqlite://

В конфиге меняем стиль и ставим автопандас, чтобы результат выводился типом DataFrame и колаб его красиво отрисовывал

In [4]:
%config SqlMagic.style = 'PLAIN_COLUMNS'
%config SqlMagic.autopandas = True

Создаем нашу таблицу из примера, который в задании

In [5]:
%%sql

CREATE TABLE returns (
    return_id INT,
    order_item_id INT,
    return_date DATE,
    reason TEXT
);

 * sqlite://
Done.


""


In [6]:
%%sql

INSERT INTO returns VALUES (7001, 5001, '2023-04-20', 'Не тот размер');
INSERT INTO returns VALUES (7002, 5003, '2023-05-15', 'Брак');
INSERT INTO returns VALUES (7003, 12345, '2024-01-05', 'бракованный');


 * sqlite://
1 rows affected.
1 rows affected.
1 rows affected.


""


In [7]:
%%sql

SELECT * FROM returns;

 * sqlite://
Done.


,return_id,order_item_id,return_date,reason
0,7001,5001,2023-04-20,Не тот размер
1,7002,5003,2023-05-15,Брак
2,7003,12345,2024-01-05,бракованный


### Рассмотрим наши два случая границ:

#### Нижняя граница
Грубая оценка, которая нам позволит просто оценить долю возвратов, где в поле `reason` встречается брак в любом виде: <br>
Брак (и все его виды написания), "какой-то" брак, брак "чего-то", т.е. что просто будет содержать "брак".

#### Верхняя граница
Здесь мы будем рассматривать случаи, где у нас уже неоднозначные ситуации. Причины по которым можно сказать, что да возврат по браку, но причина не указана открыто, как "брак".
Это случаи:
- Дефект/Сломано/Повреждено/Кривое [вставить необходимое] (в зависимости от товара, если это одежда, то это будут различные дефекты текстиля. Дыры, порезы, сломанная молния, как примеры)
- "Содержит" дефект
- Не работает / Не функционирует / Не включается / Не соответствует указанному виду товара и тп
- Трещина "где-то" (в зависимости от товара, если это обувь, то "трещина на подошве", как пример)

В данной границе очень много примеров и нужно проводить классификацию того, относится эта причина к "браку" или это уже другая причина, допустим поврежденние при доставке уже будет иной причиной, хотя технически можно все еще отнести к браку. Но для такого анализа нужно более обширный спектр данных.

### Случаи

#### Нижняя граница

SQLite плохо работает с кириллицей и из-за этого возникает проблема того, что по поиску `%брак%` он находит только бракованный

In [8]:
%%sql

SELECT
    COUNT(*) * 1.0 / (SELECT COUNT(*) FROM returns) AS lower_bound
FROM returns
WHERE LOWER(reason) LIKE LOWER('%брак%');

 * sqlite://
Done.


,lower_bound
0,0.333333


Как код выглядел бы для psql

```
CREATE TABLE returns (
    return_id INT,
    order_item_id INT,
    return_date DATE,
    reason TEXT
);

INSERT INTO returns VALUES
(7001, 5001, '2023-04-20', 'Не тот размер'),
(7002, 5003, '2023-05-15', 'Брак'),
(7003, 12345, '2024-01-05', 'бракованный');

SELECT
    COUNT(*) * 1.0 / (SELECT COUNT(*) FROM returns) AS lower_bound
FROM returns
WHERE reason ILIKE '%брак%';
```



Из-за проблемы SQLite далее решение будет на python

In [9]:
df = %sql SELECT * FROM returns
df.head()

 * sqlite://
Done.


,return_id,order_item_id,return_date,reason
0,7001,5001,2023-04-20,Не тот размер
1,7002,5003,2023-05-15,Брак
2,7003,12345,2024-01-05,бракованный


In [10]:
df['reason'] = df['reason'].str.lower()
df

,return_id,order_item_id,return_date,reason
0,7001,5001,2023-04-20,не тот размер
1,7002,5003,2023-05-15,брак
2,7003,12345,2024-01-05,бракованный


In [11]:
import pandas as pd

lower = df["reason"].str.contains(r"брак", case=False, na=False).mean()
df_result = pd.DataFrame({
    'lower': [lower]
})
df_result

,lower
0,0.666667


#### Верхняя граница

Для показательности я добавлю в таблицу еще данные

In [12]:
%%sql

INSERT INTO returns VALUES (7004, 5004, '2023-06-10', 'Заводской брак');
INSERT INTO returns VALUES (7005, 5005, '2023-06-18', 'Дефект корпуса');
INSERT INTO returns VALUES (7006, 5006, '2023-07-02', 'Не работает');
INSERT INTO returns VALUES (7007, 5007, '2023-07-15', 'Поврежден при доставке');
INSERT INTO returns VALUES (7008, 5008, '2023-08-01', 'Трещина на экране');
INSERT INTO returns VALUES (7009, 5009, '2023-08-20', 'Сломан замок');

INSERT INTO returns VALUES (7010, 5010, '2023-09-05', 'Не подошел размер');
INSERT INTO returns VALUES (7011, 5011, '2023-09-12', 'Не понравился цвет');
INSERT INTO returns VALUES (7012, 5012, '2023-09-18', 'Передумал');
INSERT INTO returns VALUES (7013, 5013, '2023-10-01', 'Нашел дешевле');
INSERT INTO returns VALUES (7014, 5014, '2023-10-10', 'Не соответствует ожиданиям');

INSERT INTO returns VALUES (7015, 5015, '2023-11-03', 'Пришел не тот товар');
INSERT INTO returns VALUES (7016, 5016, '2023-11-15', 'Неполная комплектация');
INSERT INTO returns VALUES (7017, 5017, '2023-12-01', 'Ошибочный заказ');
INSERT INTO returns VALUES (7018, 5018, '2023-12-20', 'Товар не подошел');
INSERT INTO returns VALUES (7019, 5019, '2024-01-10', 'Есть дефект');
INSERT INTO returns VALUES (7020, 5020, '2024-01-25', 'Брак упаковки');

 * sqlite://
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.


""


Решение на psql


```
SELECT
    COUNT(*) * 1.0 / (SELECT COUNT(*) FROM returns) AS upper_bound
FROM returns
WHERE reason ILIKE '%брак%'
   OR reason ILIKE '%дефект%'
   OR reason ILIKE '%слом%'
   OR reason ILIKE '%работ%'
   OR reason ILIKE '%повреж%'
   OR reason ILIKE '%трещ%'
   OR reason ILIKE '%не тот товар%';
```



Решение на python:pandas

In [13]:
df = %sql SELECT * FROM returns
df.shape

 * sqlite://
Done.


(20, 4)

In [14]:
lower = df["reason"].str.contains(r"брак", case=False, na=False).mean()

pattern = r"брак|дефект|слом|работ|повреж|трещ|не тот товар"

upper = df["reason"].str.contains(pattern, case=False, na=False).mean()

df_result['upper'] = [upper]
df_result['lower'] = [lower]

df_result

,lower,upper
0,0.2,0.55


## Задача 2

Существует гипотеза, что клиенты могут не дожидаться (и, следовательно, не забирать) свои заказы в пунктах выдачи из-за очередей, которые возникают в часы пик. Товары приходится отвозить обратно, и мы теряем деньги. Мы не умеем отслеживать напрямую очереди в пунктах выдачи. Предложите, как грубо можно оценить эффект от того, что мы научимся работать с очередями (например, выводить доп сотрудников или перебрасывать заказ в другой ближайший пункт выдачи без очереди и пр.)


### Проведение A/B-тестирования по пунктам выдачи

Провести A/B-тестирования на разных точках/районах.

То есть собрать выборки по следующим вариантам:

#### 1. Выводить дополнительных сотрудников

Простой, грубый кластерный A/B-тест.

Берем небольшое число пунктов выдачи заказов и в части из них увеличиваем количество сотрудников в часы пик. Далее по этим выборкам считаем статистику и делаем вывод о влиянии на долю товаров, которые не были выкуплены.

Так как по стоимости эксперимента нет возможности масштабировать тест на большое число точек, мы вынуждены ограничиться малым количеством ПВЗ. В этом случае нельзя корректно опираться на ЦПТ, поскольку число кластеров слишком мало.

Поэтому анализ стоит проводить на уровне агрегированных значений по ПВЗ. Возможные подходы:
- t-статистика по средним значениям на уровне кластеров (ПВЗ)
- либо бутстрап по кластерам, который менее чувствителен к малому числу наблюдений и распределению данных

#### 2. Перебрасывать заказ в другой ближайший пункт выдачи


Разбор этого варианта начинается не с того, что мы получим, а с анализа общей ситуации. По большей части здесь есть накладка с тем, что, в сравнении с доп. сотрудниками, проблема очередей может решиться, но если мы будем распределять заказы, есть вероятность ухудшить ситуацию: мы можем перебросить товар, а человек этого не заметит, и тогда возникнет путаница с ПВЗ у клиентов. Также появляются дополнительные траты на распределение заказов по пунктам выдачи.

Но в целом не отличается от первого, кроме того как мы собираем данные, если там было по конкретным точкам, то тут нужно будет брать комплексы из нескольких ПВЗ




Если делать тестовый пример, то для ввода доп сотрудников будет следующее

In [15]:
# Агрегированный уровень ПВЗ (для A/B анализа)
df_pvz = pd.DataFrame([
    {"pvz_id": "PVZ_1", "treatment": "control", "total_orders": 2000, "unpicked_orders": 120},
    {"pvz_id": "PVZ_2", "treatment": "test",    "total_orders": 1800, "unpicked_orders": 70},
    {"pvz_id": "PVZ_3", "treatment": "control", "total_orders": 2100, "unpicked_orders": 150},
    {"pvz_id": "PVZ_4", "treatment": "test",    "total_orders": 1900, "unpicked_orders": 65},
    {"pvz_id": "PVZ_5", "treatment": "control", "total_orders": 1950, "unpicked_orders": 130},
    {"pvz_id": "PVZ_6", "treatment": "test",    "total_orders": 1850, "unpicked_orders": 76},
    {"pvz_id": "PVZ_7", "treatment": "control", "total_orders": 2050, "unpicked_orders": 119},
    {"pvz_id": "PVZ_8", "treatment": "test",    "total_orders": 2000, "unpicked_orders": 74},
    {"pvz_id": "PVZ_9", "treatment": "control", "total_orders": 1980, "unpicked_orders": 123},
    {"pvz_id": "PVZ_10","treatment": "test",    "total_orders": 1820, "unpicked_orders": 66},
])

И у нас идет две гипотезы

H0: среднее количество не выкупленных товаров не отличается

H1: Отличается

посчитаем по ним t-статистику и сделаем вывод

In [16]:
from scipy import stats

alpha = 0.05

df_pvz["unpickup_rate"] = df_pvz["unpicked_orders"] / df_pvz["total_orders"]

control = df_pvz[df_pvz['treatment']=='control']["unpickup_rate"].values
test = df_pvz[df_pvz['treatment']=='test']["unpickup_rate"].values

t_crit = stats.t.ppf(1 - alpha/2, df=8)
t_stat, p_val = stats.ttest_ind(control, test, equal_var=False)

result = pd.DataFrame({
    "Control mean:": [control.mean()],
    "Test mean:": [test.mean()],
    "t-stat:": [t_stat],
    't-crit': [t_crit],
    "p-value:": [p_val]
})
result

,Control mean:,Test mean:,t-stat:,t-crit,p-value:
0,0.063653,0.037489,9.751117,2.306004,0.000084


Для этого примера получаем что (t_stat > t_crit) | (p_val < 0.05) => различия значимы

## Задача 3

Предположим, что один из брендов на площадке падает год к году по продажам. Перечислите, какие данные могут быть полезны для анализа причин падения, и предложите последовательность действий / список гипотез для выяснения причин.

У нас бренд падает от года к году по продажам -> нужно смотреть на его времянной ряд. Я бы выбрал основу анализа его падения тренд и чуть меньше внимания уделял сезонности и случайным отклонениям.

Тренд бренда можно разложить на некоторые "слои" от которых он может зависеть:
- спрос на сам бренд
- видимость бренда на площадке
- конверсия от клика на площадке
- наличие и цена
- аудитория и её изменения
- внешняя конкуренция и ассортиментная роль бренда


### Самые важные данные по этим темам

#### 1. Метрики воронки

Сюда входят показатели карточек товаров/листингов бренда, клики на эти товары, как часто этот бренд выбирают в фильтрах, CTR, конверсия в заказ, как часто наш бренд выкупают/возвращают/отменяют (если речь про возвраты, то нужно подумать про качество бренда или иные причины отказов), доля повторных покупок у этого бренда, как часто пользователи хотят к нему возвращаться.

#### 2. Видимость на площадке

Позиция в списке. Чаще всего начальная страница (обычная или страница поиска) выдает товары с сортировкой по популярности, то есть нужно посмотреть, как часто этот бренд находится вверху и как мы можем поместить его выше, чтобы повысить спрос на него. Как часто этот бренд попадает в подборки или рекомендации пользователей. Допустим, у Lamoda есть ИИ-подборка аутфитов — можем посмотреть, как часто этот бренд появляется у пользователей и как часто они его выбирают.

#### 3. Аудитория и сегменты

Нужно определить, какой возраст у целевой аудитории этого бренда, потому что более старшее поколение вряд ли склоняется к онлайн-шопингу, и при этом сам бренд не интересует молодую аудиторию сайта. Также важно учитывать контекст бренда: если, допустим, это очень дорогой бренд, но чисто визуально/эстетически он ничего интересного не может предложить более платежеспособной аудитории, а им интересуются лишь пользователи, которые не могут его себе позволить, можно сравнить средний чек выкупа пользователя и то, какой товар он просматривал.

#### 4. Ассортимент и роль бренда в категории

Насколько большой ассортимент у нашего бренда, т.е. насколько больше возможностей попасть в ленту пользователя/подборки и в целом насколько много этот бренд захватывает пользователей. Также можно рассмотреть вариант того, насколько наш бренд устойчив к фильтрации пользователей, то есть если пользователь настраивает большое количество фильтров под себя, наш бренд не отлетит. Также нужно посмотреть на долю SKU и на вывод старых SKU. Так же можно попробовать сместить всякими акциями/скидками наш бренд в более дешевый сегмент, чтобы повысить продажи, а вместе с ними и его узнаваемость. Также насколько наш бренд замещаем другими брендами, и если замещаем, то как часто делают выбор в его пользу.


### Действия / гипотезы, которые я бы предложил
---

#### 1. Бренд стал хуже виден в поиске и рекомендациях.

Можно исправить карточки товара, возможно неправильное использование фильтров для товаров, не отображается в рекомендациях

H0: Бренд все также нормально виден в рекомендациях, изменений не было.

---
#### 2. Снизился рекламный охват или эффективность рекламных размещений.

 Продвигать в рекламных баннерах и смотреть на эффект

 H0: Продвижение не сработало

 ---
#### 3. Бренд стал дороже относительно конкурентов.

Можно ввести акции/скидки на товары этого бренда

H0: Акции не увеличели продажи

---
#### 4. Ухудшилась доступность товара: меньше SKU в наличии, больше out-of-stock.

Договориться с брендом о новых поставках, вводить рекламу новинок от бренда

H0: Пополнение стока и реклама не помогла

---
#### 5. Упало качество карточек: фото, контент, рейтинг, отзывы.

Если фото, то переделать и посмотреть эффект стал ли товар продаваться больше

Если контент, то переписать и посмотреть привлекает ли он пользователей

Если рейтинг/отзывы смотреть что стало фактором понижения рейтинга стоимость/качество/итд

---
#### 6. Бренд потерял ключевую аудиторию или изменился состав покупателей.

Начать широкую таргетацию, чтобы найти новых покупателей для этого бренда

---
#### 7. Ассортимент бренда сузился или сместился в менее продающие SKU.

Начать рекламу менее продающих SKU

Либо пополнить лидирующие SKU

---
#### 8. Конкуренты забрали долю через промо, цену или лучшую выдачу.

Можно начать маркетинговую борьбу

Можно искать нецелевую аудиторию конкурентов и работать на неё

Можно выкупить места в рекомендациях / подборках

---
#### 9. Падение связано не со спросом на бренд, а с изменением структуры категории.

Изменить карточки товаров под новые реалии

И также изменить маркетинговую стратегию внутри категорий

---
#### 10. Бренд перестал попадать в актуальный пользовательский сценарий, хотя раньше был релевантен.

Изменить алгоритм подборки в сценариях или измениться самим под актуальные сценарии

---

## Задача 4

Перед вами таблица с информацией о клиентах

 Таблица users (пользователи):
| user_id | registration_date | first_order_date  | first_buying_date | last_order_date | last_buying_date |
|-----------|------------------------|-------------------------|------------------------|-----------------------|-------------------------|
| 1           | 2023-01-15           | 2023-01-20            | 2023-05-10           | 2024-11-10          | 2024-08-17            |
| 2           | 2023-03-15           | 2023-03-15            | 2023-03-15           | 2023-03-15          | 2023-03-15            |
| 3           | 2022-12-10           | 2022-12-10            | 2022-12-10           | 2026-05-15          | 2026-04-30            |
| 4…

Придумайте правило, по которому клиентов можно было бы размечать как «еще живущих» с нами и «уже окончательно оттекших». На какие метрики полезно посмотреть, чтобы правило отражало реальность? Напишите sql-запрос, который разметит клиентов согласно придуманному вами правилу.


### Правило определения оттока

Клиент считается **оттёкшим**, если время с момента его последней активности превышает половину его исторического срока жизни, но составляет не менее 90 дней.

Под историческим сроком жизни понимается период между первой и последней активностью клиента.

Таким образом, порог оттока рассчитывается по формуле:

```text
threshold = max(90 дней, lifetime / 2)
```

#### Обоснование

В качестве уровня доверия к клиенту используется половина его исторического срока жизни. Логика заключается в том, что чем дольше клиент взаимодействовал с сервисом, тем больше вероятность его возвращения после периода неактивности.

При этом вводится минимальный порог в 90 дней, чтобы не признавать новых пользователей оттёкшими слишком рано.



### Метрики для валидации правила

После разметки клиентов необходимо проверить, насколько правило соответствует реальному поведению пользователей.

#### 1. Коэффициент восстановления

Доля клиентов, которые:

были размечены как churn
но сделали покупку/заказ позже

---

#### 2. Time-to-churn распределение

Проверяем:

через сколько дней после последней активности клиент попадает в churn

Важно:

нет ли «скачка» около 90 дней (искусственный порог)
нет ли перекоса по новым/старым клиентам
---

#### 3. Время до следующей активности

После разметки полезно посмотреть, через сколько дней пользователь возвращается.

Ожидаемое поведение:

```text
alive   → возвращаются относительно быстро
churned → либо не возвращаются, либо возвращаются крайне редко
```

---

#### 4. False Churn Rate

Необходимо оценить долю клиентов, которых мы ошибочно признали оттёкшими.

Формула:

```text
False Churn Rate =
вернувшиеся churned /
все churned
```

Чем ниже значение метрики, тем точнее правило.

---

#### 5. Precision (точность оттока) через будущую активность

Берём исторический срез:

помечаем клиентов как churn на дату T
смотрим, сколько из них не вернулись за следующие N дней (например, 60–90)


In [17]:
%%sql

CREATE TABLE users (
    user_id INT,
    registration_date DATE,
    first_order_date DATE,
    first_buying_date DATE,
    last_order_date DATE,
    last_buying_date DATE
);

 * sqlite://
Done.


""


In [18]:
%%sql

INSERT INTO users VALUES (1, '2023-01-15','2023-01-20','2023-05-10','2024-11-10','2024-08-17');
INSERT INTO users VALUES (2, '2023-03-15','2023-03-15','2023-03-15','2023-03-15','2023-03-15');
INSERT INTO users VALUES (3, '2022-12-10','2022-12-10','2022-12-10','	2026-05-15','2026-04-30');

 * sqlite://
1 rows affected.
1 rows affected.
1 rows affected.


""


In [19]:
%%sql

WITH user_activity AS (
    SELECT
        user_id,

        MIN(
            COALESCE(first_order_date, first_buying_date),
            COALESCE(first_buying_date, first_order_date)
        ) AS first_activity_date,

        MAX(
            COALESCE(last_order_date, last_buying_date),
            COALESCE(last_buying_date, last_order_date)
        ) AS last_activity_date

    FROM users
)

SELECT
    user_id,

    CASE
        WHEN (
            julianday('now') -
            julianday(last_activity_date)
        ) >
        MAX(
            90,
            (
                julianday(last_activity_date) -
                julianday(first_activity_date)
            ) / 2
        )
        THEN 'churned'
        ELSE 'alive'
    END AS status

FROM user_activity;


 * sqlite://
Done.


,user_id,status
0,1,churned
1,2,churned
2,3,alive
